In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

panel = pd.read_parquet("Panel.parquet")

HAR  = ["RV_daily", "RV_weekly", "RV_monthly"]
HARX = HAR + ["CDS_level", "CDS_change_5d", "VIX", "Yield_slope", "BAA10Y"]

TEST_YEARS = [2020, 2021, 2022, 2023, 2024]

RF_PARAMS  = dict(n_estimators=300, max_features=0.5, random_state=42, n_jobs=-1)
XGB_PARAMS = dict(n_estimators=300, max_depth=4, learning_rate=0.05,
                  subsample=0.8, random_state=42, verbosity=0)

print(f"Loaded {len(panel):,} rows, {panel['Firm'].nunique()} firms")

Loaded 211,196 rows, 56 firms


In [2]:
def dm_test(actual, pred_bench, pred_chal, h):
    d = (actual - pred_bench) ** 2 - (actual - pred_chal) ** 2
    T = len(d)
    if T < 30:
        return np.nan, np.nan
    dbar = d.mean()
    gamma0 = np.var(d, ddof=0)
    # autocovariances up to lag h-1. forecasts h steps ahead overlap.
    gamma = sum((1 - k / h) * np.cov(d[k:], d[:-k])[0, 1]
                for k in range(1, h)) if h > 1 else 0
    var = (gamma0 + 2 * gamma) / T
    if var <= 0:
        return np.nan, np.nan
    stat = dbar / np.sqrt(var)
    # HLN correction
    hln  = np.sqrt((T + 1 - 2 * h + h * (h - 1) / T) / T) 
    stat *= hln
    p = 2 * stats.t.sf(abs(stat), df=T - 1)
    return stat, p

In [3]:
def firm_predictions(panel, tgt, challenger_feats, kind):
    d = panel.dropna(subset=HARX + [tgt]).copy()
    recs = []
    for ty in TEST_YEARS:
        tr = d[d["Date"].dt.year <  ty]
        te = d[d["Date"].dt.year == ty]
        if len(te) == 0:
            continue
        ytr = tr[tgt].values
        ph = np.clip(LinearRegression().fit(tr[HAR].values, ytr).predict(te[HAR].values), 0, None)
        if kind == "lin":
            pc = np.clip(LinearRegression().fit(tr[challenger_feats].values, ytr).predict(te[challenger_feats].values), 0, None)
        elif kind == "rf":
            pc = np.clip(RandomForestRegressor(**RF_PARAMS).fit(tr[challenger_feats].values, ytr).predict(te[challenger_feats].values), 0, None)
        else:
            pc = np.clip(xgb.XGBRegressor(**XGB_PARAMS).fit(tr[challenger_feats].values, ytr).predict(te[challenger_feats].values), 0, None)
        tmp = te[["Firm", "Sector"]].copy()
        tmp["Actual"], tmp["Bench"], tmp["Chal"] = te[tgt].values, ph, pc
        recs.append(tmp)
    return pd.concat(recs)

In [4]:
CHALLENGERS = [("HARX", HARX, "lin"), ("RF", HARX, "rf"), ("XGB", HARX, "xgb")]
HORIZONS = [("Target_h1", 1, "1-day"), ("Target_h5", 5, "1-week"), ("Target_h22", 22, "1-month")]

dm_results = []
for tgt, h, hlabel in HORIZONS:
    for name, feats, kind in CHALLENGERS:
        preds = firm_predictions(panel, tgt, feats, kind)
        better = worse = nosig = 0
        for _, g in preds.groupby("Firm"):
            stat, p = dm_test(g["Actual"].values, g["Bench"].values, g["Chal"].values, h)
            if np.isnan(p):
                continue
            if p < 0.05 and stat > 0: better += 1
            elif p < 0.05 and stat < 0: worse += 1
            else: nosig += 1
        dm_results.append({"Horizon": hlabel, "Model": name,
                           "Better": better, "Worse": worse, "No_diff": nosig})
        print(f"{hlabel:8} {name:5} | better {better:2}  worse {worse:2}  no diff {nosig:2}")

dm_results = pd.DataFrame(dm_results)
dm_results.to_csv("DM_Tests.csv", index=False)

1-day    HARX  | better  6  worse  7  no diff 42
1-day    RF    | better  0  worse 21  no diff 34
1-day    XGB   | better  0  worse 12  no diff 43
1-week   HARX  | better  4  worse 10  no diff 41
1-week   RF    | better  0  worse 16  no diff 39
1-week   XGB   | better  0  worse 23  no diff 32
1-month  HARX  | better  3  worse 15  no diff 37
1-month  RF    | better  0  worse 53  no diff  2
1-month  XGB   | better  0  worse 53  no diff  2


In [5]:
sector_records = []
for tgt, h, hlabel in HORIZONS:
    preds = firm_predictions(panel, tgt, HARX, "lin")
    for firm, g in preds.groupby("Firm"):
        stat, p = dm_test(g["Actual"].values, g["Bench"].values, g["Chal"].values, h)
        if np.isnan(p):
            continue
        sector_records.append({"Horizon": hlabel, "Sector": g["Sector"].iloc[0], "Firm": firm,
                               "Better": int(p < 0.05 and stat > 0),
                               "Worse":  int(p < 0.05 and stat < 0)})

sector_df = pd.DataFrame(sector_records)
sector_summary = (sector_df.groupby(["Horizon", "Sector"])[["Better", "Worse"]].sum().reset_index())
sector_summary.to_csv("DM_By_Sector.csv", index=False)

# In order of leverage
SEC_ORDER = ["XLF", "XLY", "XLE", "XLI", "XLK", "XLV"]
for hlabel in ["1-day", "1-week", "1-month"]:
    print(f"\n{hlabel}:")
    sub = sector_summary[sector_summary["Horizon"] == hlabel].set_index("Sector")
    for sec in SEC_ORDER:
        if sec in sub.index:
            print(f"  {sec}: {sub.loc[sec,'Better']} better, {sub.loc[sec,'Worse']} worse")


1-day:
  XLF: 0 better, 0 worse
  XLY: 4 better, 0 worse
  XLE: 1 better, 0 worse
  XLI: 0 better, 1 worse
  XLK: 1 better, 1 worse
  XLV: 0 better, 5 worse

1-week:
  XLF: 0 better, 0 worse
  XLY: 2 better, 1 worse
  XLE: 1 better, 0 worse
  XLI: 0 better, 3 worse
  XLK: 1 better, 1 worse
  XLV: 0 better, 5 worse

1-month:
  XLF: 0 better, 1 worse
  XLY: 2 better, 1 worse
  XLE: 0 better, 0 worse
  XLI: 0 better, 4 worse
  XLK: 1 better, 2 worse
  XLV: 0 better, 7 worse
